# LoRA for the local AI (Google Colab)

Trains a LoRA for Ollama's `llama3.2:3b` on the project's two writing jobs (swarm explanations, title summaries), using a free Colab GPU.

1. A larger model (Qwen2.5-7B-Instruct) answers each training prompt.
2. The project's own checks (`local_ai_tuning.py`) keep only answers that pass.
3. A LoRA is trained on Llama-3.2-3B-Instruct, the model Ollama runs as `llama3.2:3b`, on those answers.

**Before you start**
- On your PC: `python local_ai_tuning.py export` makes `autonomous/ai-tuning/train-inputs.jsonl`.
- On Hugging Face: accept the license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct (approval is usually quick), and make a **read** token at https://huggingface.co/settings/tokens.
- In Colab: key icon (Secrets) in the left bar → add `HF_TOKEN` with that token, and allow this notebook to use it.
- **Runtime → Change runtime type → T4 GPU.**

Time on a T4: about 30–45 minutes for answers, 15–25 minutes for training.

What leaves your PC: `train-inputs.jsonl`, which holds synthetic sequence facts and public paper titles only.

In [ ]:
# 1. Check the GPU is there. If this fails: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Project code and libraries (Colab already has PyTorch).
!git clone --depth 1 https://github.com/DNA-Blockchain/Helloworld.git /content/network-os
%cd /content/network-os/colab
!pip install -q -U transformers peft accelerate bitsandbytes
import sys; sys.path.insert(0, "/content/network-os/colab")
import local_ai_lora as ll

In [ ]:
# 3. Hugging Face login, for the Llama 3.2 weights (from the HF_TOKEN secret).
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
# 4. Upload train-inputs.jsonl from autonomous/ai-tuning/ on your PC.
from google.colab import files
name = next(iter(files.upload()))
rows = ll.load_rows(name)
print(len(rows), "prompts:", {t: sum(r["task"] == t for r in rows) for t in ("explain", "summary")})

In [ ]:
# 5. The teacher answers every prompt; the project's checks keep only answers that pass.
#    Failed prompts are retried twice with sampling. Loaded in 4-bit to fit the T4.
teacher, tokenizer = ll.load_model(ll.TEACHER, four_bit=True)
kept, rejected = ll.build_dataset(rows, teacher, tokenizer, retries=2, batch_size=16)
import os; os.makedirs("nos-lora", exist_ok=True)
ll.save_rows("nos-lora/kept.jsonl", kept)
ll.save_rows("nos-lora/rejected.jsonl", rejected)
del teacher
ll.release_gpu_memory()

In [ ]:
# 6. READ THESE before training. The checks catch invented numbers, health words, made-up framing and
#    missing key facts, but not every mistake. If the answers look wrong, stop here.
import collections, random
print("kept:", collections.Counter(r["task"] for r in kept), " rejected:", collections.Counter(r["task"] for r in rejected))
print("why rejected (last attempt):", collections.Counter(f.split(":")[0] for r in rejected for f in r["attempts"][-1]["flags"]).most_common(6))
random.seed(0)
for task in ("explain", "summary"):
    for r in random.sample([k for k in kept if k["task"] == task], k=min(4, sum(k["task"] == task for k in kept))):
        print(f"\n[{task}] IN: {r['input'] if task == 'summary' else chr(10).join(r['input'])}")
        print(f"      OUT: {r['answer']}")

In [ ]:
# 7. Train the LoRA on the kept answers only (the loss covers the answer, not the prompt).
adapter = ll.train_lora(kept, "nos-lora", epochs=2, rank=16, learning_rate=2e-4, batch_size=4, grad_accum=2)

In [ ]:
# 8. The trained model on the 8 test sequences it never saw, scored by the same checks.
results = ll.try_adapter(adapter, ll.test_rows())
print(f"passed {sum(s['ok'] for _, _, s in results)}/{len(results)} (llama3.2:3b alone: 0/8; nos-explain: 8/8)")
for row, answer, score in results:
    print(f"\n{row['id']}  {'ok' if score['ok'] else 'BAD ' + '; '.join(score['flags'])}\n  {answer}")

In [ ]:
# 9. Download the adapter (~100 MB) with the kept and rejected answers.
!zip -qr nos-lora.zip nos-lora -x "nos-lora/checkpoints/*" && ls -lh nos-lora.zip
files.download("nos-lora.zip")

## Back on your PC

Unzip `nos-lora.zip` into the project's `ollama/` folder so the adapter is at `ollama/nos-lora/adapter/`, then:

```
python local_ai_tuning.py create                  # also builds nos-explain-lora and nos-summary-lora
python local_ai_tuning.py run --variants system,lora
```

Keep the LoRA only if it scores better than `system` on your PC, and read its outputs in `autonomous/ai-tuning/`.